In [1]:
!pip install -qq --upgrade pandas tqdm bitsandbytes feedparser sentence-transformers umap-learn xlsxwriter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 require

In [2]:
import requests
import feedparser
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
from tqdm import tqdm
import re

from concurrent.futures import ThreadPoolExecutor, as_completed

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

import umap
import hdbscan
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from sklearn.metrics import davies_bouldin_score

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

session = requests.Session()

2026-05-20 09:06:57.888837: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779268018.408359      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779268018.539938      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779268019.545834      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779268019.545883      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779268019.545886      22 computation_placer.cc:177] computation placer alr

In [3]:
TOPIC_RSS = {

    "world": [
        "https://vnexpress.net/rss/the-gioi.rss",
        "https://thanhnien.vn/rss/the-gioi.rss",
        "https://tuoitre.vn/rss/the-gioi.rss",
        "https://dantri.com.vn/rss/the-gioi.rss",
        "https://infonet.vietnamnet.vn/rss/the-gioi.rss"
    ],

    "sports": [
        "https://vnexpress.net/rss/the-thao.rss",
        "https://thanhnien.vn/rss/the-thao.rss",
        "https://tuoitre.vn/rss/the-thao.rss",
        "https://dantri.com.vn/rss/the-thao.rss",
        "https://infonet.vietnamnet.vn/rss/the-thao.rss"
    ],

    "technology": [
        "https://vnexpress.net/rss/so-hoa.rss",
        "https://thanhnien.vn/rss/cong-nghe.rss",
        "https://tuoitre.vn/rss/nhip-song-so.rss",
        "https://dantri.com.vn/rss/cong-nghe.rss",
        "https://infonet.vietnamnet.vn/rss/cong-nghe.rss"
    ],

    "business": [
        "https://vnexpress.net/rss/kinh-doanh.rss",
        "https://thanhnien.vn/rss/tai-chinh-kinh-doanh.rss",
        "https://tuoitre.vn/rss/kinh-doanh.rss",
        "https://dantri.com.vn/rss/kinh-doanh.rss",
        "https://infonet.vietnamnet.vn/rss/kinh-doanh.rss"
    ],

    "entertainment": [
        "https://vnexpress.net/rss/giai-tri.rss",
        "https://thanhnien.vn/rss/giai-tri.rss",
        "https://tuoitre.vn/rss/giai-tri.rss",
        "https://dantri.com.vn/rss/giai-tri.rss",
        "https://infonet.vietnamnet.vn/rss/giai-tri.rss"
    ],

    "education": [
        "https://vnexpress.net/rss/giao-duc.rss",
        "https://thanhnien.vn/rss/giao-duc.rss",
        "https://tuoitre.vn/rss/giao-duc.rss",
        "https://dantri.com.vn/rss/giao-duc.rss",
        "https://infonet.vietnamnet.vn/rss/giao-duc.rss"
    ]
}

In [4]:
def collect_articles(topic_rss, limit_per_feed=20):

    articles = []

    for topic, rss_list in topic_rss.items():

        for rss_url in rss_list:

            try:

                feed = feedparser.parse(rss_url)

                for entry in feed.entries[:limit_per_feed]:

                    articles.append({

                        "topic": topic,
                        "rss_source": rss_url,
                        "title": entry.get("title", ""),
                        "url": entry.get("link", ""),
                        "published": entry.get("published", "")

                    })

            except Exception as e:

                print("RSS error:", rss_url)

    return pd.DataFrame(articles)

In [5]:
rss_df = collect_articles(
    TOPIC_RSS,
    limit_per_feed=50
)

rss_df.info()
rss_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1411 entries, 0 to 1410
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   topic       1411 non-null   str  
 1   rss_source  1411 non-null   str  
 2   title       1411 non-null   str  
 3   url         1411 non-null   str  
 4   published   1411 non-null   str  
dtypes: str(5)
memory usage: 419.7 KB


,topic,rss_source,title,url,published
0,world,https://vnexpress.net/rss/the-gioi.rss,"Ông Trump cho xây căn cứ drone, bệnh viện ngầm...",https://vnexpress.net/ong-trump-cho-xay-can-cu...,"Wed, 20 May 2026 15:30:42 +0700"
1,world,https://vnexpress.net/rss/the-gioi.rss,Nụ hôn của ông Putin định hướng cuộc sống của ...,https://vnexpress.net/nu-hon-cua-ong-putin-din...,"Wed, 20 May 2026 15:23:34 +0700"
2,world,https://vnexpress.net/rss/the-gioi.rss,Quân đội Myanmar tái chiếm thị trấn giáp biên ...,https://vnexpress.net/quan-doi-myanmar-tai-chi...,"Wed, 20 May 2026 15:13:10 +0700"
3,world,https://vnexpress.net/rss/the-gioi.rss,Mỹ muốn Ukraine 'chuyển giao công nghệ drone',https://vnexpress.net/my-muon-ukraine-chuyen-g...,"Wed, 20 May 2026 15:04:44 +0700"
4,world,https://vnexpress.net/rss/the-gioi.rss,Nga - Trung cần gì ở nhau?,https://vnexpress.net/nga-trung-can-gi-o-nhau-...,"Wed, 20 May 2026 13:00:00 +0700"


In [6]:
def parse_vnexpress(soup):

    title = soup.select_one("h1.title-detail")
    description = soup.select_one("p.description")
    paragraphs = soup.select("article.fck_detail p.Normal")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [7]:
def parse_thanhnien(soup):

    title = soup.select_one("h1.detail-title")
    description = soup.select_one("h2.detail-sapo")
    paragraphs = soup.select(".detail-content > :is(p, h1, h2, h3, h4, h5, h6)")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [8]:
def parse_tuoitre(soup):

    title = soup.select_one("h1.article-title")
    description = soup.select_one("h2.detail-sapo")
    paragraphs = soup.select(".detail-content > :is(p, h1, h2, h3, h4, h5, h6)")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [9]:
def parse_dantri(soup):

    title = soup.select_one("h1.e-magazine__title")
    description = soup.select_one("h2.e-magazine__sapo")
    paragraphs = soup.select(".e-magazine__body > :is(p, h1, h2, h3, h4, h5, h6)")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [10]:
def parse_vietnamnet(soup):

    title = soup.select_one("h1.contentDetail-title")
    description = soup.select_one(".contentDetail-sapo")
    paragraphs = soup.select(".contentDetail__main-reading > :is(p, h1, h2, h3, h4, h5, h6)")

    return {
        "title": title.text.strip() if title else "",
        "description": description.text.strip() if description else "",
        "content": re.sub(r"\s+", " ", "\n".join([
            p.get_text(" ", strip=True)
            for p in paragraphs
        ]))
    }

In [11]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

def get_article_content(url):
    res = session.get(
        url,
        headers=headers,
        timeout=20
    )

    res.encoding = "utf-8"

    soup = BeautifulSoup(res.text, "html.parser")

    if "vnexpress.net" in url:
        parsed = parse_vnexpress(soup)

    elif "thanhnien.vn" in url:
        parsed = parse_thanhnien(soup)

    elif "tuoitre.vn" in url:
        parsed = parse_tuoitre(soup)

    elif "dantri.com.vn" in url:
        parsed = parse_dantri(soup)

    elif "infonet.vietnamnet.vn" in url:
        parsed = parse_vietnamnet(soup)

    else:
        return None

    parsed["url"] = url

    parsed["full_text"] = "\n".join([
        parsed["title"],
        parsed["description"],
        parsed["content"]
    ])

    parsed["summary"] = "\n".join([
        parsed["title"],
        parsed["description"],
        parsed["content"][:1000]
    ])

    return parsed

In [12]:
def fetch_article(row):

    try:

        article = get_article_content(row["url"])

        if article and len(article["full_text"]) > 300:

            article["topic"] = row["topic"]
            article["rss_source"] = row["rss_source"]

            return article

    except Exception:

        return None

In [13]:
data = []

rows = list(rss_df.to_dict("records"))

with ThreadPoolExecutor(max_workers=64) as executor:

    futures = [
        executor.submit(fetch_article, row)
        for row in rows
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        result = future.result()

        if result is not None:
            data.append(result)

df = pd.DataFrame(data)

print(df.shape)

100%|██████████| 1411/1411 [02:40<00:00,  8.80it/s]

(1070, 8)


In [14]:
embed_model = SentenceTransformer(
    "BAAI/bge-m3"
)

embeddings = embed_model.encode(
    df["summary"].tolist(),
    show_progress_bar=True,
    batch_size=128
)

print(embeddings.shape)

embeddings = normalize(embeddings, norm="l2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

(1070, 1024)


In [15]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    metric='euclidean'
)

labels = clusterer.fit_predict(embeddings)

df["cluster"] = labels

In [16]:
mask = labels != -1

score = silhouette_score(
    embeddings[mask],
    labels[mask],
    metric="cosine"
)

print(f"Silhouette score: {score}")

from hdbscan import validity_index

score = validity_index(
    embeddings[mask].astype(np.float64),
    labels[mask],
    metric='euclidean'
)

print(f"DBCV: {score}")

db_index = davies_bouldin_score(embeddings[mask], labels[mask])
print(f"Davies-Bouldin Index: {db_index}")

noise_ratio = np.mean(labels == -1)
print(f"Noise ratio: {noise_ratio}")

def cluster_coherence(cluster_embs):

    centroid = cluster_embs.mean(axis=0)

    sims = cosine_similarity(
        cluster_embs,
        [centroid]
    )

    return sims.mean()

cluster_scores = []

for cid in np.unique(labels):

    # bỏ noise
    if cid == -1:
        continue

    cluster_embs = embeddings[labels == cid]

    # cluster phải có >=2 point
    if len(cluster_embs) < 2:
        continue

    score = cluster_coherence(cluster_embs)

    cluster_scores.append({
        "cluster": cid,
        "size": len(cluster_embs),
        "coherence": score
    })

# sort theo coherence
cluster_scores = sorted(
    cluster_scores,
    key=lambda x: x["coherence"],
    reverse=True
)

# preview
for row in cluster_scores[:10]:
    print(row)

weighted_mean = np.average(
    [x["coherence"] for x in cluster_scores],
    weights=[x["size"] for x in cluster_scores]
)

print(f"Weighted mean: {weighted_mean}")

Silhouette score: 0.3390652537345886
DBCV: 0.16036969949172325
Davies-Bouldin Index: 1.648302031298298
Noise ratio: 0.7261682242990655
{'cluster': np.int64(19), 'size': 6, 'coherence': np.float32(0.91504)}
{'cluster': np.int64(9), 'size': 6, 'coherence': np.float32(0.90789694)}
{'cluster': np.int64(21), 'size': 6, 'coherence': np.float32(0.90684444)}
{'cluster': np.int64(4), 'size': 5, 'coherence': np.float32(0.87676173)}
{'cluster': np.int64(15), 'size': 10, 'coherence': np.float32(0.86766165)}
{'cluster': np.int64(6), 'size': 7, 'coherence': np.float32(0.84988743)}
{'cluster': np.int64(14), 'size': 14, 'coherence': np.float32(0.8487944)}
{'cluster': np.int64(5), 'size': 17, 'coherence': np.float32(0.8432529)}
{'cluster': np.int64(16), 'size': 7, 'coherence': np.float32(0.83120495)}
{'cluster': np.int64(3), 'size': 5, 'coherence': np.float32(0.83067703)}
Weighted mean: 0.8129737619247046


/usr/local/lib/python3.12/dist-packages/hdbscan/validity.py:30: RuntimeWarning: overflow encountered in power
  distance_matrix[distance_matrix != 0] = (1.0 / distance_matrix[
/usr/local/lib/python3.12/dist-packages/hdbscan/validity.py:30: RuntimeWarning: overflow encountered in power
  distance_matrix[distance_matrix != 0] = (1.0 / distance_matrix[
/usr/local/lib/python3.12/dist-packages/hdbscan/validity.py:30: RuntimeWarning: overflow encountered in power
  distance_matrix[distance_matrix != 0] = (1.0 / distance_matrix[
/usr/local/lib/python3.12/dist-packages/hdbscan/validity.py:30: RuntimeWarning: overflow encountered in power
  distance_matrix[distance_matrix != 0] = (1.0 / distance_matrix[
/usr/local/lib/python3.12/dist-packages/hdbscan/validity.py:30: RuntimeWarning: overflow encountered in power
  distance_matrix[distance_matrix != 0] = (1.0 / distance_matrix[
/usr/local/lib/python3.12/dist-packages/hdbscan/validity.py:30: RuntimeWarning: overflow encountered in power
  distance

In [17]:
for cluster_id in sorted(df["cluster"].unique()):

    print("=" * 80)
    print("CLUSTER", cluster_id)

    subset = df[df["cluster"] == cluster_id]

    for title in subset["title"].head(5):
        print("-", title)

CLUSTER -1
- Ấn Độ muốn thúc đẩy hợp tác quốc phòng với Việt Nam
- Cựu thủ tướng Đức 'nuối tiếc' vì châu Âu từ chối đối thoại với Nga
- Rào cản khiến nhiều nhà trẻ Mỹ không được bóc chuối
- Tiêm kích Su-57 hai chỗ ngồi lần đầu bay thử
- Mưa lũ cuốn trôi cầu, nhấn chìm ôtô ở Trung Quốc
CLUSTER 0
- Phim của IU bị chỉ trích 'sai lệch lịch sử'
- Bất chấp phản đối, IU vẫn bỏ túi 105 tỉ đồng từ dự án 'Perfect Crown'?
- IU và Byeon Woo Seok xin lỗi sau khi bom tấn bị tố 'sai lệch lịch sử'
- Bom tấn của IU bị chỉ trích vì 'sai lệch lịch sử'
- Đạo diễn Perfect Crown cúi đầu xin lỗi vì tranh cãi xuyên tạc lịch sử
CLUSTER 1
- 'Cô gái đẹp nhất thế giới' diện váy tôn đường cong ở Cannes
- Hương Giang: 'Sẵn sàng chinh phục Miss Grand International'
- Vợ sắp cưới Ronaldo gợi cảm ở Cannes
- Hoa hậu Hòa bình diện đầm 'Nước mắt màn đêm' của Nguyễn Minh Tuấn
- Hương Giang thi diễn áo tắm tại Miss Grand International
CLUSTER 2
- Hồng Ngọc: 'Tôi chọn cách sống chậm'
- Cuộc sống của Hồng Nhung trong penthou

In [18]:
import pandas as pd

cluster_dict = {}

for cluster_id in sorted(df["cluster"].unique()):

    subset = df[df["cluster"] == cluster_id]

    cluster_dict[f"CLUSTER {cluster_id}"] = subset["title"].tolist()

# padding để các cột dài bằng nhau
max_len = max(len(v) for v in cluster_dict.values())

for k in cluster_dict:
    cluster_dict[k] += [""] * (max_len - len(cluster_dict[k]))

# tạo bảng
out_df = pd.DataFrame(cluster_dict)

# export excel
out_df.to_excel("clusters.xlsx", index=False)


with pd.ExcelWriter("clusters.xlsx", engine="xlsxwriter") as writer:
    out_df.to_excel(writer, index=False, sheet_name="Clusters")

    workbook = writer.book
    worksheet = writer.sheets["Clusters"]

    # wrap text
    wrap_format = workbook.add_format({
        "text_wrap": True,
        "valign": "top"
    })

    worksheet.set_column(0, len(out_df.columns)-1, 40, wrap_format)

    # freeze header
    worksheet.freeze_panes(1, 0)

In [19]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [20]:
def generate_batch(prompts, batch_size=4):

    results = []

    system_prompt = (
        "Bạn là AI chuyên tổng hợp xu hướng báo chí.\n"
        "Chỉ trả về đúng 1 câu summary.\n"
        "Không giải thích.\n"
        "Không bullet point.\n"
        "Không nhắc lại yêu cầu."
    )

    for i in tqdm(range(0, len(prompts), batch_size)):

        batch_prompts = prompts[i:i + batch_size]

        # ==========================================
        # BUILD CHAT MESSAGES
        # ==========================================

        message_batch = []

        for prompt in batch_prompts:

            messages = [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ]

            message_batch.append(messages)

        # ==========================================
        # APPLY CHAT TEMPLATE
        # ==========================================

        text_batch = tokenizer.apply_chat_template(
            message_batch,
            tokenize=False,
            add_generation_prompt=True
        )

        # ==========================================
        # TOKENIZE
        # ==========================================

        inputs = tokenizer(
            text_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)

        # ==========================================
        # GENERATE
        # ==========================================

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

        # ==========================================
        # REMOVE PROMPT TOKENS
        # ==========================================

        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids
            in zip(inputs.input_ids, outputs)
        ]

        # ==========================================
        # DECODE
        # ==========================================

        decoded = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        results.extend([
            d.strip()
            for d in decoded
        ])

        torch.cuda.empty_cache()

    return results

In [21]:


# =========================================================
# CONFIG
# =========================================================

TOP_K = 5
DIVERSITY_THRESHOLD = 0.9

# =========================================================
# HELPER
# =========================================================

def normalize(v):
    norm = np.linalg.norm(v)

    if norm == 0:
        return v

    return v / norm


def select_representative_docs(
    cluster_embeddings,
    cluster_indices,
    top_k=5,
    diversity_threshold=0.9
):
    """
    Chọn top-k bài:
    - gần centroid
    - có diversity nhẹ
    """

    centroid = np.mean(cluster_embeddings, axis=0)
    centroid = normalize(centroid)

    normalized_embs = np.array([
        normalize(x)
        for x in cluster_embeddings
    ])

    sims = cosine_similarity(
        [centroid],
        normalized_embs
    )[0]

    sorted_idx = np.argsort(-sims)

    selected_local_idx = []

    for idx in sorted_idx:

        current_emb = normalized_embs[idx]

        too_similar = False

        for selected_idx in selected_local_idx:

            sim = cosine_similarity(
                [current_emb],
                [normalized_embs[selected_idx]]
            )[0][0]

            if sim >= diversity_threshold:
                too_similar = True
                break

        if not too_similar:
            selected_local_idx.append(idx)

        if len(selected_local_idx) >= top_k:
            break

    selected_global_indices = [
        cluster_indices[i]
        for i in selected_local_idx
    ]

    return selected_global_indices


def build_trend_prompt(representative_articles):

    prompt = """Tóm tắt các bài báo sau thành đúng 1 câu tiếng Việt ngắn.

Không bullet point.
Không giải thích.
Không nhắc lại yêu cầu.

Các bài báo:
"""

    for i, article in enumerate(representative_articles, 1):

        prompt += f"""
{i}.
Tiêu đề: {article["title"]}

Tóm tắt:
{article["description"]}
"""
    
    prompt += "\nKết quả:\n"
    return prompt


# =========================================================
# BUILD CLUSTER DATA
# =========================================================

cluster_data = {}

cluster_docs = defaultdict(list)

for idx, row in df.iterrows():
    cluster_docs[row["cluster"]].append(idx)

for cluster_id, indices in cluster_docs.items():
    # skip noise
    if cluster_id == -1:
        continue

    subset = df.iloc[indices]

    cluster_embs = embeddings[indices]

    # =====================================================
    # SELECT REPRESENTATIVE ARTICLES
    # =====================================================

    representative_indices = select_representative_docs(
        cluster_embs,
        indices,
        top_k=TOP_K,
        diversity_threshold=DIVERSITY_THRESHOLD
    )

    representative_articles = []
    for idx in representative_indices:
        row = df.iloc[idx]

        representative_articles.append({
            "index": int(idx),
            "title": row["title"],
            "description": row["description"],
            "url": row["url"],
            "topic": row["topic"]
        })

    # =====================================================
    # ALL ARTICLES
    # =====================================================

    all_articles = []

    for idx in indices:
        row = df.iloc[idx]

        all_articles.append({
            "index": int(idx),
            "title": row["title"],
            "description": row["description"],
            "url": row["url"],
            "topic": row["topic"]
        })

    # =====================================================
    # BUILD PROMPT
    # =====================================================

    prompt = build_trend_prompt(
        representative_articles
    )

    # =====================================================
    # SAVE CLUSTER
    # =====================================================

    cluster_data[cluster_id] = {
        "cluster_id": int(cluster_id),
        "num_articles": len(indices),
        "article_indices": indices,
        "representative_indices": representative_indices,
        "representative_articles": representative_articles,
        "all_articles": all_articles,
        "prompt": prompt,
        "trend": None
    }

# =========================================================
# GENERATE TRENDS
# =========================================================

cluster_ids = list(cluster_data.keys())

prompts = [
    cluster_data[cid]["prompt"]
    for cid in cluster_ids
]

trends = generate_batch(
    prompts,
    batch_size=32
)

for cid, trend in zip(cluster_ids, trends):
    cluster_data[cid]["trend"] = trend
    
# =========================================================
# SORT
# =========================================================

sorted_clusters = sorted(
    cluster_data.items(),
    key=lambda x: x[1]["num_articles"],
    reverse=True
)

cluster_data = {
    i: data
    for i, (_, data) in enumerate(sorted_clusters)
}

# =========================================================
# PREVIEW
# =========================================================

for cid, data in cluster_data.items():
    print("=" * 80)
    print("CLUSTER:", cid)

    print("TREND:")
    print(data["trend"])

    print(f"\nNUMBER OF ARTICLES: {data["num_articles"]}")

    print("\nREPRESENTATIVE ARTICLES:")
    for article in data["representative_articles"]:

        print("-", article["title"])


  0%|          | 0/1 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

100%|██████████| 1/1 [00:28<00:00, 28.91s/it]

CLUSTER: 0
TREND:
Man City bị Bournemouth cầm hòa, Arsenal giành chức vô địch Ngoại hạng Anh 2025-2026.

NUMBER OF ARTICLES: 44

REPRESENTATIVE ARTICLES:
- Guardiola chúc mừng Arsenal vô địch
- Arsenal chính thức vô địch nước Anh sau hơn 20 năm chờ đợi
- Man City hết cửa vô địch Ngoại hạng Anh
- HLV Pep Guardiola: 'Arsenal xứng đáng với chức vô địch'
- Man City tự ‘bắn vào chân’, Arsenal được dâng sớm chức vô địch Ngoại hạng Anh
CLUSTER: 1
TREND:
Lính dù Nga dùng súng ngắn bắn hạ UAV Ukraine, đồng thời quân đội Nga sử dụng tên lửa bắn nổ xe tăng Abrams và trạm Starlink của Ukraine, trong khi Ukraine sử dụng UAV tấn công xe tăng T-90M và BMPT "Terminator" của

NUMBER OF ARTICLES: 32

REPRESENTATIVE ARTICLES:
- Khoảnh khắc lính dù Nga bắn hạ UAV 'khủng' của Ukraine
- Video UAV Ukraine tập kích xe tăng T-90 và xe bọc thép hiếm của Nga
- Video quân đội Nga bắn cháy xe tăng Abrams thứ 4 của Ukraine
- Khoảnh khắc UAV cảm tử Nga làm nổ tung trạm liên lạc Starlink của Ukraine
- FPV Nga truy đu

In [ ]:
import json
import numpy as np
from datetime import datetime

export_trends = []

for rank, data in cluster_data.items():
    export_trends.append({
        "rank": int(rank),
        "cluster_id": int(data["cluster_id"]),
        "trend_name": data["trend"],
        "article_count": int(data["num_articles"]),
        "trend": data["trend"],
        "representative_articles": data["representative_articles"],
        "articles": data["all_articles"]
    })

payload = {
    "project_name": "News Trend Detection",
    "generated_at": datetime.now().isoformat(),
    "model": {
        "embedding_model": "BAAI/bge-m3",
        "llm_model": "Qwen/Qwen2.5-7B-Instruct",
        "quantization": "4-bit"
    },
    "metrics": {
        "total_rss_articles": int(len(rss_df)),
        "total_scraped_articles": int(len(df)),
        "num_clusters": int(len([x for x in np.unique(labels) if x != -1])),
        "noise_ratio": float(np.mean(labels == -1)),
        "weighted_coherence": float(weighted_mean)
    },
    "trends": export_trends
}

with open("trends.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Exported trends.json")